In [7]:
import librosa
import numpy as np
import pandas as pd
import soundfile as sf
import os
import random

In [8]:
ESC50_AUDIO_PATH = "../data/ESC-50"
ESC50_METADATA_PATH = "../data/ESC-50/esc50.csv"

BIO_CLASSES = [
    'dog', 'chirping_birds', 'crow', 'frog', 'cow', 'hen', 'insects', 'sheep', 
    'pig', 'rooster', 'cat', 'crickets', 'crying_baby', 'breathing', 
    'coughing', 'sneezing', 'snoring', 'laughing', 'clapping'
]

GEO_CLASSES = [
    'sea_waves', 'rain', 'wind', 'thunderstorm', 'water_drops', 'crackling_fire'
]

ANTRO_CLASSES = [
    'helicopter', 'chainsaw', 'siren', 'car_horn', 'engine', 'train', 'church_bells', 
    'airplane', 'fireworks', 'hand_saw', 'keyboard_typing', 'mouse_click', 
    'footsteps', 'door_wood_knock', 'door_wood_creaks', 'clock_alarm', 
    'clock_tick', 'glass_breaking', 'brushing_teeth', 'toilet_flush', 
    'washing_machine', 'vacuum_cleaner', 'can_opening', 'drinking_sipping',
    'pouring_water'
]

In [9]:
SAMPLE_RATE = 32000
DURATION_SEC = 5 
DURATION_SAMPLES = SAMPLE_RATE * DURATION_SEC

N_SAMPLES_PER_SCENE = 1000

metadada_lits = []

scene_types = {
        0: 'bio_dominated',
        1: 'antro_dominated',
        2: 'geo_dominated',
    }

scene_alphas = {
        0: (10, 1, 1), # Biofonia
        1: (1, 10, 1), # Antropofonia
        2: (1, 1, 10), # Geofonia
    }


In [10]:
def load_audio(file_path, target_samples):

    signal, sr = librosa.load(file_path, sr=SAMPLE_RATE, mono=True)
    
    if len(signal) < target_samples:
        # preenche com silêncio
        padding = target_samples - len(signal)
        signal = np.pad(signal, (0, padding), 'constant')
    elif len(signal) > target_samples:
        # trunca
        signal = signal[:target_samples]
        
    return signal


In [11]:
def normalize_rms(signal):
    epsilon = 1e-10
    rms = np.sqrt(np.mean(signal**2))
    target_rms = 0.1 
    return signal * (target_rms / (rms + epsilon))

In [12]:
def build_file_lists():
    dicio = {}

    df_esc50 = pd.read_csv(ESC50_METADATA_PATH)
    for index, row in df_esc50.iterrows():
        file_path = os.path.join(ESC50_AUDIO_PATH, row["filename"])
        file_path = file_path.replace("\\", "/")
        category = row["category"]
        
        if category in BIO_CLASSES:
            dicio.setdefault(category, []).append(file_path)

        elif category in ANTRO_CLASSES:
            dicio.setdefault(category, []).append(file_path)

        elif category in GEO_CLASSES:
            dicio.setdefault(category, []).append(file_path)

            
    return dicio

In [13]:
def generate_mix(proportions, map, bio, antro, geo, output_filename):
    print(f"\nGerando mix para: {output_filename}")
    print(f"Proporções: {bio}-{proportions[0]} | {antro}-{proportions[1]} | {geo}-{proportions[2]}")
    
    final_signal = np.zeros(DURATION_SAMPLES)
    
    # Adiciona Biofonia
    if proportions[0] > 0:
        bio_list = map[bio]
        file_to_load = random.choice(bio_list)
        signal = load_audio(file_to_load, DURATION_SAMPLES)
        signal_norm = normalize_rms(signal)
        final_signal += signal_norm * proportions[0]
        print(f"  + Áudio BIO:   {os.path.basename(file_to_load)}")

    # Adiciona Antropofonia
    if proportions[1] > 0:
        antro_list = map[antro]
        file_to_load = random.choice(antro_list)
        signal = load_audio(file_to_load, DURATION_SAMPLES)
        signal_norm = normalize_rms(signal)
        final_signal += signal_norm * proportions[1]
        print(f"  + Áudio ANTRO: {os.path.basename(file_to_load)}")

    # Adiciona Geofonia
    if proportions[2] > 0:
        geo_list = map[geo]
        file_to_load = random.choice(geo_list)
        signal = load_audio(file_to_load, DURATION_SAMPLES)
        signal_norm = normalize_rms(signal)
        final_signal += signal_norm * proportions[2]
        print(f"  + Áudio GEO:   {os.path.basename(file_to_load)}")
        
    # Normaliza o sinal final para evitar clipping 
    max_val = np.max(np.abs(final_signal))
    if max_val > 1.0:
        final_signal = final_signal / max_val
    
    # Salva 
    sf.write(output_filename, final_signal, SAMPLE_RATE)

In [14]:
map = build_file_lists()
for key in map.keys():
    if key in BIO_CLASSES:
        print(f"{key} - BIO")
    elif key in ANTRO_CLASSES:
        print(f'{key} - ANTRO')
    else:
        print(f'{key} - GEO')    

dog - BIO
chirping_birds - BIO
vacuum_cleaner - ANTRO
thunderstorm - GEO
door_wood_knock - ANTRO
can_opening - ANTRO
crow - BIO
clapping - BIO
fireworks - ANTRO
chainsaw - ANTRO
airplane - ANTRO
mouse_click - ANTRO
pouring_water - ANTRO
train - ANTRO
sheep - BIO
water_drops - GEO
church_bells - ANTRO
clock_alarm - ANTRO
keyboard_typing - ANTRO
wind - GEO
footsteps - ANTRO
frog - BIO
cow - BIO
brushing_teeth - ANTRO
car_horn - ANTRO
crackling_fire - GEO
helicopter - ANTRO
drinking_sipping - ANTRO
rain - GEO
insects - BIO
laughing - BIO
hen - BIO
engine - ANTRO
breathing - BIO
crying_baby - BIO
hand_saw - ANTRO
coughing - BIO
glass_breaking - ANTRO
snoring - BIO
toilet_flush - ANTRO
pig - BIO
washing_machine - ANTRO
clock_tick - ANTRO
sneezing - BIO
rooster - BIO
sea_waves - GEO
siren - ANTRO
cat - BIO
door_wood_creaks - ANTRO
crickets - BIO


In [ ]:
OUTPUT_AUDIO_DIR = "./output_synthetic_scenes_2"
os.makedirs(OUTPUT_AUDIO_DIR, exist_ok=True)

for scene_label, scene_name in scene_types.items():
    print(60 * "=")
    print(f"{scene_name}")
    
    for i in range(N_SAMPLES_PER_SCENE):
        
        dominant_prop = random.uniform(0.6, 0.9)
        remaining_prop = 1.0 - dominant_prop
        
        proporcoes = np.random.dirichlet(scene_alphas[scene_label])
        print(f"  Amostra {i+1}/{N_SAMPLES_PER_SCENE}: {proporcoes}")
        
        filename = f"scene{scene_label}_{scene_name}_{i:04d}.wav"
        output_path = os.path.join(OUTPUT_AUDIO_DIR, filename)
        
        bio = random.choice(BIO_CLASSES)
        antro = random.choice(ANTRO_CLASSES)
        geo = random.choice(GEO_CLASSES)
        
        generate_mix(proporcoes, map, bio, antro, geo, output_path)
        
        metadada_lits.append({
            "filename": filename,
            "scene_label": scene_label,
            "scene_name": scene_name,
            "bio_class": bio,
            "antro_class": antro,
            "geo_class": geo,
            "bio_proportion": proporcoes[0],
            "antro_proportion": proporcoes[1],
            "geo_proportion": proporcoes[2]
        })

df_metadata = pd.DataFrame(metadada_lits)
df_metadata.to_csv(os.path.join(OUTPUT_AUDIO_DIR, "metadata_synthetic_scenes.csv"), index=False)

bio_dominated
  Amostra 1/1000: [0.79047357 0.15429542 0.055231  ]

Gerando mix para: ./output_synthetic_scenes_2\scene0_bio_dominated_0000.wav
Proporções: crickets-0.790473573770917 | drinking_sipping-0.15429542463239715 | sea_waves-0.05523100159668589
  + Áudio BIO:   4-194246-A-13.wav
  + Áudio ANTRO: 1-67230-A-29.wav
  + Áudio GEO:   5-219379-C-11.wav
  Amostra 2/1000: [0.89235708 0.05649242 0.0511505 ]

Gerando mix para: ./output_synthetic_scenes_2\scene0_bio_dominated_0001.wav
Proporções: breathing-0.8923570832675464 | engine-0.056492417306294264 | rain-0.05115049942615945
  + Áudio BIO:   5-261464-A-23.wav
  + Áudio ANTRO: 3-115382-A-44.wav
  + Áudio GEO:   1-26222-A-10.wav
  Amostra 3/1000: [0.84618057 0.03661291 0.11720652]

Gerando mix para: ./output_synthetic_scenes_2\scene0_bio_dominated_0002.wav
Proporções: sheep-0.8461805714276764 | washing_machine-0.03661290648872483 | crackling_fire-0.11720652208359883
  + Áudio BIO:   2-119161-B-8.wav
  + Áudio ANTRO: 4-218199-H-35.wav